# LiTS Liver Tumor — Boundary Loss Ablation

비교: `ce_dice` / `plwce_dice` (baseline) vs `ce_dice_boundary` / `plwce_dice_boundary` / `plwce_boundary`

In [1]:
# === Cell 0: 환경 설정 ===
import subprocess, sys
for pkg in ['segmentation-models-pytorch', 'nibabel', 'openpyxl', 'kagglehub', 'scipy', 'optuna']:
    subprocess.check_call([sys.executable, '-m', 'pip', 'install', pkg, '-q'])
import os, warnings, json, random, glob
warnings.filterwarnings('ignore')
os.environ['TQDM_DISABLE'] = '1'
import numpy as np, nibabel as nib, cv2
from tqdm import tqdm
import torch, torch.nn as nn, torch.optim as optim
from torch.utils.data import Dataset, DataLoader
import matplotlib; matplotlib.use('Agg')
import matplotlib.pyplot as plt
from sklearn.model_selection import train_test_split
import segmentation_models_pytorch as smp
sys.path.insert(0, '/root/imbalanced-data-LWCE/medical_data')
from custom_losses import get_loss_function

DOMAIN = 'lits'; NUM_CLASSES = 3; CLASS_NAMES = ['Background', 'Liver', 'Tumor']
IMG_SIZE = 256; BATCH_SIZE = 16; NUM_WORKERS = 0; SEED = 42
HU_MIN, HU_MAX = -100, 250
random.seed(SEED); np.random.seed(SEED); torch.manual_seed(SEED)
if torch.cuda.is_available(): torch.cuda.manual_seed_all(SEED)
RESULTS_DIR = '/root/imbalanced-data-LWCE/medical_data/boundary_ablation/results/lits'
os.makedirs(RESULTS_DIR, exist_ok=True)
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f'Device: {device}'); print('환경 설정 완료')

Device: cuda
환경 설정 완료


In [2]:
# === Cell 1: 데이터 로드 ===
import kagglehub
raw_path = kagglehub.dataset_download('andrewmvd/liver-tumor-segmentation')
print('Dataset path:', raw_path)

vol_files  = sorted(glob.glob(os.path.join(raw_path, '**', 'volume-*.nii*'), recursive=True))
mask_files = sorted(glob.glob(os.path.join(raw_path, '**', 'segmentation-*.nii*'), recursive=True))
print(f'Volumes: {len(vol_files)}  Masks: {len(mask_files)}')

def get_vol_idx(fp):
    base = os.path.basename(fp)
    return int(''.join(filter(str.isdigit, base.split('.')[0])))

vol_dict  = {get_vol_idx(fp): fp for fp in vol_files}
mask_dict = {get_vol_idx(fp): fp for fp in mask_files}
common_idx = sorted(set(vol_dict.keys()) & set(mask_dict.keys()))
vol_mask_pairs = [(vol_dict[i], mask_dict[i]) for i in common_idx]
print(f'매칭된 CT 볼륨 수: {len(vol_mask_pairs)}')

SLICE_DIR = '/tmp/lits_slices'
os.makedirs(SLICE_DIR, exist_ok=True)

def hu_window_normalize(arr):
    arr = np.clip(arr, HU_MIN, HU_MAX)
    return ((arr - HU_MIN) / (HU_MAX - HU_MIN)).astype(np.float32)

existing_slices = glob.glob(os.path.join(SLICE_DIR, '*.npz'))
if len(existing_slices) < 100:
    print('슬라이스 전처리 중...')
    for vol_path, mask_path in tqdm(vol_mask_pairs, desc='Processing volumes'):
        idx = get_vol_idx(vol_path)
        vol_arr  = nib.load(vol_path).get_fdata().astype(np.float32)
        mask_arr = nib.load(mask_path).get_fdata().astype(np.int64)
        vol_arr  = hu_window_normalize(vol_arr)
        for s in range(vol_arr.shape[2]):
            if mask_arr[:, :, s].max() == 0:
                continue
            img_s  = cv2.resize(vol_arr[:, :, s],  (IMG_SIZE, IMG_SIZE), interpolation=cv2.INTER_LINEAR)
            mask_s = cv2.resize(mask_arr[:, :, s], (IMG_SIZE, IMG_SIZE), interpolation=cv2.INTER_NEAREST).astype(np.int64)
            np.savez_compressed(
                os.path.join(SLICE_DIR, f'vol{idx:03d}_s{s:04d}.npz'),
                image=img_s.astype(np.float32), label=mask_s)
    print('슬라이스 전처리 완료')
else:
    print(f'캐시 사용: {len(existing_slices)}개')

slice_files = sorted(glob.glob(os.path.join(SLICE_DIR, '*.npz')))
print(f'총 슬라이스 수 (liver 포함): {len(slice_files)}')

# 볼륨 단위 8:1:1 분할 (데이터 누수 방지)
vol_ids = sorted(set(int(os.path.basename(f).split('_')[0][3:]) for f in slice_files))
tr_ids, tmp_ids = train_test_split(vol_ids, test_size=0.2, random_state=SEED)
val_ids, test_ids = train_test_split(tmp_ids, test_size=0.5, random_state=SEED)
tr_set, val_set, test_set = set(tr_ids), set(val_ids), set(test_ids)
def get_vol_id(f): return int(os.path.basename(f).split('_')[0][3:])
tr_files   = [f for f in slice_files if get_vol_id(f) in tr_set]
val_files  = [f for f in slice_files if get_vol_id(f) in val_set]
test_files = [f for f in slice_files if get_vol_id(f) in test_set]
print(f'Train: {len(tr_files)}  Val: {len(val_files)}  Test: {len(test_files)}')

class LiTSDataset(Dataset):
    def __init__(self, npz_files, augment=False):
        self.files = npz_files; self.augment = augment
    def __len__(self): return len(self.files)
    def __getitem__(self, idx):
        d = np.load(self.files[idx])
        image = d['image'].astype(np.float32)
        label = d['label'].astype(np.int64)
        if self.augment:
            if random.random() > 0.5: image = np.fliplr(image).copy(); label = np.fliplr(label).copy()
            if random.random() > 0.5: image = np.flipud(image).copy(); label = np.flipud(label).copy()
        image = np.stack([image, image, image], axis=0).astype(np.float32)
        return torch.from_numpy(image), torch.from_numpy(label)

train_loader = DataLoader(LiTSDataset(tr_files, augment=True),
                          batch_size=BATCH_SIZE, shuffle=True, num_workers=NUM_WORKERS, pin_memory=True)
val_loader   = DataLoader(LiTSDataset(val_files, augment=False),
                          batch_size=BATCH_SIZE, shuffle=False, num_workers=NUM_WORKERS, pin_memory=True)
test_loader  = DataLoader(LiTSDataset(test_files, augment=False),
                          batch_size=BATCH_SIZE, shuffle=False, num_workers=NUM_WORKERS, pin_memory=True)

print('클래스 비율 계산 중...')
class_counts = np.zeros(NUM_CLASSES, dtype=np.int64)
for fp in tqdm(tr_files):
    label = np.load(fp)['label'].astype(np.int64)
    for c in range(NUM_CLASSES):
        class_counts[c] += int((label == c).sum())
class_counts = class_counts.tolist()
total = sum(class_counts)
for c, (name, cnt) in enumerate(zip(CLASS_NAMES, class_counts)):
    print(f'  [{c}] {name:<12}: {cnt:>12,} ({100*cnt/total:.4f}%)')
print(f'BG : Liver  = {class_counts[0] / class_counts[1]:.1f} : 1')
print(f'BG : Tumor  = {class_counts[0] / class_counts[2]:.1f} : 1')
print(f'class_counts = {class_counts}')
print('DataLoader 완료')

Extracting files...
Dataset path: /root/.cache/kagglehub/datasets/andrewmvd/liver-tumor-segmentation/versions/5
Volumes: 51  Masks: 131
매칭된 CT 볼륨 수: 51
슬라이스 전처리 중...
슬라이스 전처리 완료
총 슬라이스 수 (liver 포함): 6802
Train: 5433  Val: 793  Test: 576
클래스 비율 계산 중...
  [0] Background  :  333,333,760 (93.6181%)
  [1] Liver       :   21,787,222 (6.1190%)
  [2] Tumor       :      936,106 (0.2629%)
BG : Liver  = 15.3 : 1
BG : Tumor  = 356.1 : 1
class_counts = [333333760, 21787222, 936106]
DataLoader 완료


In [3]:
# === Cell 2: 모델 정의 ===

def build_model():
    return smp.UnetPlusPlus(
        encoder_name='resnet50', encoder_weights='imagenet',
        in_channels=3, classes=NUM_CLASSES, activation=None,
    ).to(device)

def compute_val_mdice(model, loader):
    """빠른 Val mDice (Liver+Tumor 평균, BG 제외)"""    
    model.eval()
    dice_per_class = np.zeros(NUM_CLASSES - 1)
    counts = np.zeros(NUM_CLASSES - 1)
    with torch.no_grad():
        for imgs, masks in loader:
            imgs, masks = imgs.to(device), masks.to(device)
            preds = torch.argmax(model(imgs), dim=1)
            for c_idx, c in enumerate(range(1, NUM_CLASSES)):
                tp = ((preds == c) & (masks == c)).sum().item()
                fp = ((preds == c) & (masks != c)).sum().item()
                fn = ((preds != c) & (masks == c)).sum().item()
                dice_per_class[c_idx] += 2 * tp / (2 * tp + fp + fn + 1e-8)
                counts[c_idx] += 1
    return float(np.mean(dice_per_class / np.maximum(counts, 1)))

def compute_val_metrics(model, loader):
    """전체 Val 지표: Liver Dice, Tumor Dice, mDice."""    
    model.eval()
    dice_per_class = np.zeros(NUM_CLASSES - 1)
    counts = np.zeros(NUM_CLASSES - 1)
    with torch.no_grad():
        for imgs, masks in loader:
            imgs, masks = imgs.to(device), masks.to(device)
            preds = torch.argmax(model(imgs), dim=1)
            for c_idx, c in enumerate(range(1, NUM_CLASSES)):
                tp = ((preds == c) & (masks == c)).sum().item()
                fp = ((preds == c) & (masks != c)).sum().item()
                fn = ((preds != c) & (masks == c)).sum().item()
                dice_per_class[c_idx] += 2 * tp / (2 * tp + fp + fn + 1e-8)
                counts[c_idx] += 1
    per_class = dice_per_class / np.maximum(counts, 1)
    return {
        'Liver_Dice': float(per_class[0]),
        'Tumor_Dice': float(per_class[1]),
        'mDice':      float(np.mean(per_class)),
    }

print('모델 + 유틸리티 함수 준비 완료')

모델 + 유틸리티 함수 준비 완료


In [4]:
# === Cell 3: 학습 함수 ===

FINAL_EPOCHS = 75
FINAL_LR     = 1e-4

def train_model(loss_name, alpha=1.0, gamma=2.0, epochs=75, lr=1e-4,
                subset_ratio=1.0, tag='', patience=15):
    model     = build_model()
    optimizer = optim.Adam(model.parameters(), lr=lr)
    scheduler = optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=epochs)
    criterion = get_loss_function(loss_name, class_counts=class_counts, alpha=alpha, gamma=gamma)
    name = f'{loss_name}_a{alpha:.2f}' if alpha != 1.0 else loss_name
    if tag: name = f'{tag}_{name}'
    print(f"\n{'='*60}\n{name}  (epochs={epochs})\n{'='*60}")
    if subset_ratio < 1.0:
        n = max(1, int(len(train_loader.dataset) * subset_ratio))
        sub_ds = torch.utils.data.Subset(
            train_loader.dataset, random.sample(range(len(train_loader.dataset)), n))
        loader = DataLoader(sub_ds, batch_size=BATCH_SIZE, shuffle=True, num_workers=NUM_WORKERS)
    else:
        loader = train_loader
    history = {'loss': [], 'val_mdice': []}
    best_mdice = 0.0
    save_path = f'/tmp/ba_lits_{name}.pth'
    patience_counter = 0
    for epoch in range(epochs):
        # --- Boundary Loss annealing: 지연 선형 (20% warmup 후 선형, 3-comp: max=1/3, 2-comp: max=0.5) ---
        if criterion.boundary_loss is not None:
            WARMUP = int(epochs * 0.2)
            has_region = criterion.region_loss is not None
            alpha_max = 1/3 if has_region else 0.5
            if epoch <= WARMUP:
                alpha_t = 0.0
            else:
                alpha_t = min((epoch - WARMUP) / (epochs - WARMUP), 1.0) * alpha_max
            criterion.set_boundary_alpha(alpha_t)
        model.train()
        epoch_loss = 0.0
        for imgs, masks in tqdm(loader, desc=f'Ep{epoch+1:02d}/{epochs}', leave=False):
            imgs, masks = imgs.to(device), masks.to(device)
            optimizer.zero_grad()
            logits = model(imgs)
            loss = criterion(logits, masks)
            loss.backward()
            optimizer.step()
            epoch_loss += loss.item()
        scheduler.step()
        avg_loss = epoch_loss / len(loader)
        val_mdice = compute_val_mdice(model, val_loader)
        history['loss'].append(avg_loss)
        history['val_mdice'].append(val_mdice)
        print(f'Ep{epoch+1:02d} | Loss: {avg_loss:.4f} | Val mDice: {val_mdice:.4f}', end='')
        if val_mdice > best_mdice:
            best_mdice = val_mdice
            torch.save(model.state_dict(), save_path)
            print('  <- Best!', end='')
            patience_counter = 0
        else:
            patience_counter += 1
        print()
        if patience_counter >= patience:
            print(f'  Early stopping at epoch {epoch+1} (patience={patience})')
            break
    model.load_state_dict(torch.load(save_path, weights_only=True))
    print(f'최고 Val mDice: {best_mdice:.4f}')
    return model, history, best_mdice

print('train_model() 준비 완료')

train_model() 준비 완료


In [5]:
# === Cell 4: Optuna — PLWCE alpha 탐색 (Boundary Loss 환경) ===
import optuna, traceback
optuna.logging.set_verbosity(optuna.logging.WARNING)
os.environ['TQDM_DISABLE'] = '1'

ALPHA_LOW    = 2.5
ALPHA_HIGH   = 15.0
PROXY_EPOCHS = 8
PROXY_RATIO  = 0.15
N_TRIALS     = 20

def make_objective(loss_name):
    def objective(trial):
        alpha = trial.suggest_float('alpha', ALPHA_LOW, ALPHA_HIGH)
        try:
            _, _, mdice = train_model(loss_name, alpha=alpha,
                                      epochs=PROXY_EPOCHS, subset_ratio=PROXY_RATIO)
            return mdice
        except Exception:
            traceback.print_exc()
            return None
    return objective

# --- plwce_dice_boundary ---
print('Optuna: plwce_dice_boundary ...')
sampler_pdb = optuna.samplers.GridSampler(
    {'alpha': np.linspace(ALPHA_LOW, ALPHA_HIGH, N_TRIALS).tolist()}
)
study_pdb = optuna.create_study(direction='maximize', sampler=sampler_pdb)
study_pdb.optimize(make_objective('plwce_dice_boundary'), n_trials=N_TRIALS)
best_trials_pdb = [t for t in study_pdb.trials if t.value is not None]
best_alpha_with_dice = (
    best_trials_pdb[int(np.argmax([t.value for t in best_trials_pdb]))].params['alpha']
    if best_trials_pdb else 7.0
)
print(f'  best alpha (with Dice): {best_alpha_with_dice:.3f}')

# --- plwce_boundary (no Dice) ---
print('Optuna: plwce_boundary (no Dice) ...')
sampler_pb = optuna.samplers.GridSampler(
    {'alpha': np.linspace(ALPHA_LOW, ALPHA_HIGH, N_TRIALS).tolist()}
)
study_pb = optuna.create_study(direction='maximize', sampler=sampler_pb)
study_pb.optimize(make_objective('plwce_boundary'), n_trials=N_TRIALS)
best_trials_pb = [t for t in study_pb.trials if t.value is not None]
best_alpha_without_dice = (
    best_trials_pb[int(np.argmax([t.value for t in best_trials_pb]))].params['alpha']
    if best_trials_pb else 10.0
)
print(f'  best alpha (no Dice):   {best_alpha_without_dice:.3f}')

optuna_data = {
    'plwce_dice_boundary': {'best_alpha': best_alpha_with_dice},
    'plwce_boundary':      {'best_alpha': best_alpha_without_dice},
}
with open(os.path.join(RESULTS_DIR, f'{DOMAIN}_boundary_optuna.json'), 'w') as f:
    json.dump(optuna_data, f, indent=2)
print('Optuna 결과 저장 완료')

Optuna: plwce_dice_boundary ...


config.json:   0%|          | 0.00/156 [00:00<?, ?B/s]

model.safetensors:   0%|          | 0.00/102M [00:00<?, ?B/s]

[plwce_dice_boundary] CE weights (plwce): Generated.
[plwce_dice_boundary] Components: CE + DiceLoss + BoundaryLoss  (λ=0.333 each)

plwce_dice_boundary_a14.34  (epochs=8)
Ep01 | Loss: 0.7484 | Val mDice: 0.3881  <- Best!
Ep02 | Loss: 0.5685 | Val mDice: 0.4048  <- Best!
Ep03 | Loss: 0.4509 | Val mDice: 0.4444  <- Best!
Ep04 | Loss: 0.3888 | Val mDice: 0.4380
Ep05 | Loss: 0.3366 | Val mDice: 0.4592  <- Best!
Ep06 | Loss: 0.3034 | Val mDice: 0.4616  <- Best!
Ep07 | Loss: 0.2784 | Val mDice: 0.4655  <- Best!
Ep08 | Loss: 0.2597 | Val mDice: 0.4693  <- Best!
최고 Val mDice: 0.4693
[plwce_dice_boundary] CE weights (plwce): Generated.
[plwce_dice_boundary] Components: CE + DiceLoss + BoundaryLoss  (λ=0.333 each)

plwce_dice_boundary_a3.16  (epochs=8)
Ep01 | Loss: 0.6147 | Val mDice: 0.4166  <- Best!
Ep02 | Loss: 0.4359 | Val mDice: 0.4475  <- Best!
Ep03 | Loss: 0.3515 | Val mDice: 0.4683  <- Best!
Ep04 | Loss: 0.2893 | Val mDice: 0.4799  <- Best!
Ep05 | Loss: 0.2466 | Val mDice: 0.4914  <- Be

In [6]:
# === Cell 5: Boundary Ablation 전체 학습 ===

# --- Optuna 결과 로드 (Cell 4 미실행 시 JSON fallback) ---
try:
    _ = best_alpha_with_dice
except NameError:
    try:
        with open(os.path.join(RESULTS_DIR, f'{DOMAIN}_boundary_optuna.json')) as f:
            d = json.load(f)
        best_alpha_with_dice    = d['plwce_dice_boundary']['best_alpha']
        best_alpha_without_dice = d['plwce_boundary']['best_alpha']
        print(f'Optuna 로드: with_dice={best_alpha_with_dice:.3f}, no_dice={best_alpha_without_dice:.3f}')
    except FileNotFoundError:
        best_alpha_with_dice    = 7.0
        best_alpha_without_dice = 10.0
        print(f'Optuna 미실행 → fallback alpha with_dice=7.0, no_dice=10.0')

experiments = [
    ('ce_dice',             1.0,                     'CE+Dice                     [baseline]'),
    ('plwce_dice',          best_alpha_with_dice,    f'PLWCE+Dice                  (α={best_alpha_with_dice:.3f}) [baseline]'),
    ('ce_dice_boundary',    1.0,                     'CE+Dice+BL                  [literature]'),
    ('plwce_dice_boundary', best_alpha_with_dice,    f'PLWCE+Dice+BL               (α={best_alpha_with_dice:.3f})'),
    ('plwce_boundary',      best_alpha_without_dice, f'PLWCE+BL     (no Dice)       (α={best_alpha_without_dice:.3f})'),
]

all_results = {}
for loss_name, alpha, label in experiments:
    model, history, best_mdice = train_model(
        loss_name=loss_name, alpha=alpha,
        epochs=FINAL_EPOCHS, lr=FINAL_LR, tag='ba')
    all_results[label] = {
        'model': model, 'history': history, 'best_mdice': best_mdice,
        'loss_name': loss_name, 'alpha': alpha,
    }

print('\n' + '='*65)
print('[Boundary Ablation 요약 — Val mDice]')
print(f"{'Loss':<52} {'Val mDice':>10}")
print('-'*64)
for label, v in all_results.items():
    print(f"{label:<52} {v['best_mdice']:>10.4f}")

[ce_dice] Components: CE + DiceLoss  (λ=0.500 each)

ba_ce_dice  (epochs=75)
Ep01 | Loss: 0.4907 | Val mDice: 0.4919  <- Best!
Ep02 | Loss: 0.2075 | Val mDice: 0.5004  <- Best!
Ep03 | Loss: 0.1070 | Val mDice: 0.4992
Ep04 | Loss: 0.0847 | Val mDice: 0.5124  <- Best!
Ep05 | Loss: 0.0658 | Val mDice: 0.4995
Ep06 | Loss: 0.0680 | Val mDice: 0.4947
Ep07 | Loss: 0.0658 | Val mDice: 0.5111
Ep08 | Loss: 0.0644 | Val mDice: 0.5095
Ep09 | Loss: 0.0561 | Val mDice: 0.5112
Ep10 | Loss: 0.0501 | Val mDice: 0.5115
Ep11 | Loss: 0.0533 | Val mDice: 0.5105
Ep12 | Loss: 0.0562 | Val mDice: 0.5135  <- Best!
Ep13 | Loss: 0.0501 | Val mDice: 0.5210  <- Best!
Ep14 | Loss: 0.0466 | Val mDice: 0.5181
Ep15 | Loss: 0.0499 | Val mDice: 0.5040
Ep16 | Loss: 0.0498 | Val mDice: 0.5003


KeyboardInterrupt: 

In [ ]:
# === Cell 6: 평가 및 결과 저장 ===
import pandas as pd

COLORS = ['#4878D0', '#EE854A', '#6ACC65', '#D65F5F', '#B47CC7']

# --- 학습 곡선 ---
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(16, 5))
for i, (label, v) in enumerate(all_results.items()):
    c = COLORS[i % len(COLORS)]
    ax1.plot(v['history']['loss'],      label=label[:40], color=c)
    ax2.plot(v['history']['val_mdice'], label=label[:40], color=c)
ax1.set_title('Training Loss'); ax1.set_xlabel('Epoch'); ax1.legend(fontsize=7)
ax2.set_title('Val mDice');     ax2.set_xlabel('Epoch'); ax2.legend(fontsize=7)
plt.tight_layout()
plt.savefig(os.path.join(RESULTS_DIR, 'boundary_ablation_curves.png'), dpi=150)
plt.show()

# --- Test set 정량 평가 ---
print('\n[Test Set 정량 평가]')
print(f"{'Loss':<52} {'mDice':>7} {'Liver':>7} {'Tumor':>7}")
print('-' * 75)

final_results = {}
for label, v in all_results.items():
    m = compute_val_metrics(v['model'], test_loader)
    final_results[label] = m
    print(f"{label:<52} {m['mDice']:>7.4f} {m['Liver_Dice']:>7.4f} {m['Tumor_Dice']:>7.4f}")

# --- 바 차트 (3 panels: Liver_Dice, Tumor_Dice, mDice) ---
labels = list(final_results.keys())
fig, axes = plt.subplots(1, 3, figsize=(18, 5))
for ax, metric in zip(axes, ['Liver_Dice', 'Tumor_Dice', 'mDice']):
    vals = [final_results[l][metric] for l in labels]
    bars = ax.bar(range(len(labels)), vals,
                  color=[COLORS[i % len(COLORS)] for i in range(len(labels))])
    ax.set_xticks(range(len(labels)))
    ax.set_xticklabels([l[:35] for l in labels], rotation=30, ha='right', fontsize=7)
    ax.set_ylabel(metric); ax.set_title(metric)
    for bar, val in zip(bars, vals):
        ax.text(bar.get_x() + bar.get_width() / 2, bar.get_height() + 0.002,
                f'{val:.4f}', ha='center', va='bottom', fontsize=7)
plt.suptitle(f'Boundary Ablation — Test Dice ({DOMAIN.upper()})', fontsize=12)
plt.tight_layout()
plt.savefig(os.path.join(RESULTS_DIR, 'boundary_ablation_bar.png'), dpi=150)
plt.show()

# --- JSON 저장 ---
with open(os.path.join(RESULTS_DIR, f'{DOMAIN}_boundary_ablation.json'), 'w') as f:
    json.dump({l: {k: v for k, v in m.items()} for l, m in final_results.items()},
              f, indent=2, ensure_ascii=False)
print(f'결과 저장 완료: {RESULTS_DIR}')

# --- Excel 저장 ---
summary_rows = []
for label, m in final_results.items():
    summary_rows.append({
        'Loss_Function': label,
        'mDice':         round(m['mDice'],      4),
        'Liver_Dice':    round(m['Liver_Dice'],  4),
        'Tumor_Dice':    round(m['Tumor_Dice'],  4),
    })
df_summary = pd.DataFrame(summary_rows)

history_rows = []
for label, v in all_results.items():
    for ep, (loss, mdice) in enumerate(
            zip(v['history']['loss'], v['history']['val_mdice']), 1):
        history_rows.append({
            'Loss_Function': label,
            'Epoch':         ep,
            'Train_Loss':    round(loss,  6),
            'Val_mDice':     round(mdice, 6),
        })
df_history = pd.DataFrame(history_rows)

excel_path = os.path.join(RESULTS_DIR, f'{DOMAIN}_boundary_ablation.xlsx')
with pd.ExcelWriter(excel_path, engine='openpyxl') as writer:
    df_summary.to_excel(writer, sheet_name='Summary',          index=False)
    df_history.to_excel(writer, sheet_name='Training_History', index=False)
print(f'Excel 저장: {excel_path}')